# Day 2 - Tokenization Practice

This notebook turns `docs/day2/theory.md` into code you can practice with.

You will implement three tokenizer styles:

1. **Whitespace tokenization** - split on spaces.
2. **Regex tokenization** - keep words, numbers, and punctuation as separate tokens.
3. **BPE-like tokenization** - learn reusable subword pieces by counting adjacent pairs.

Then you will compare token counts and OOV behavior on small text first, and on Tiny Shakespeare / WikiText-2 if the Hugging Face `datasets` package can download them.

How to use this notebook:

- Run cells top to bottom.
- Fill in each `TODO` before running its check cell.
- The early checks use tiny examples from the theory doc, so you can verify them by hand.
- The real-data cells are optional. If downloads fail, the notebook falls back to built-in sample text.

## Setup

Run the install cell if `datasets` or `tokenizers` is missing. If you already have them installed, it should finish quickly.

In [3]:
%pip install datasets tokenizers

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
from collections import Counter
import re
import random

random.seed(42)

## Tiny Texts First

Before touching real datasets, use text small enough to inspect with your eyes. The point is to see exactly where each tokenizer cuts.

In [5]:
TRAIN_TEXTS = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "hug pug pun bun hugs",
    "I don't know New York, but I know tokenization.",
    "Numbers like 12345 and 10000 can be awkward for tokenizers.",
]

TEST_TEXTS = [
    "the bug sat on the rug",
    "How many r's are in strawberry?",
    "A new username vikrantGPT2026 appeared in New York.",
    "Try 987654 + 12345 without a calculator.",
]

print("train chars:", sum(len(t) for t in TRAIN_TEXTS))
print("test chars: ", sum(len(t) for t in TEST_TEXTS))

train chars: 170
test chars:  144


## TODO 1 - Whitespace Tokenization

The simplest tokenizer: split wherever Python sees whitespace. This collapses repeated spaces and newlines automatically if you use the right string method.

In [6]:
def whitespace_tokenize(text):
    """Return a list of tokens by splitting on whitespace."""
    return text.split()


In [7]:
assert whitespace_tokenize("  the   cat\n sat  ") == ["the", "cat", "sat"]
assert whitespace_tokenize("cat.") == ["cat."]
print("CHECK 1 passed - whitespace tokenizer works")

CHECK 1 passed - whitespace tokenizer works


## TODO 2 - Regex Tokenization

Whitespace tokenization glues punctuation to words. A simple regex tokenizer can do better by returning words, numbers, and punctuation separately.

This is still rule-based. It is useful, but it is not the final modern LLM answer.

In [8]:
TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\w\s]")

def regex_tokenize(text):
    """Return words, numbers, and punctuation as separate tokens."""
    return TOKEN_RE.findall(text)

In [9]:
assert regex_tokenize("the cat sat.") == ["the", "cat", "sat", "."]
assert regex_tokenize("Don't split 12345!") == ["Don't", "split", "12345", "!"]
print("CHECK 2 passed - regex tokenizer separates punctuation")

CHECK 2 passed - regex tokenizer separates punctuation


## Compare the First Two Tokenizers

Look at the output, not just the count. The important lesson is where the tokenizer cuts.

In [10]:
sample = "I don't know New York, but strawberry has letters and 12345 has digits."

for name, tokenizer in [
    ("whitespace", whitespace_tokenize),
    ("regex", regex_tokenize),
]:
    tokens = tokenizer(sample)
    print(f"{name:10} {len(tokens):2} tokens -> {tokens}")

whitespace 13 tokens -> ['I', "don't", 'know', 'New', 'York,', 'but', 'strawberry', 'has', 'letters', 'and', '12345', 'has', 'digits.']
regex      15 tokens -> ['I', "don't", 'know', 'New', 'York', ',', 'but', 'strawberry', 'has', 'letters', 'and', '12345', 'has', 'digits', '.']


## TODO 3 - Word-Level Vocabulary and OOV

A word-level tokenizer needs a fixed vocabulary. Anything outside it becomes `<unk>`.

Implement a small vocabulary builder, then measure how often test tokens become unknown.

In [11]:
UNK = "<unk>"

def build_word_vocab(tokenized_texts, max_size):
    """Build a vocabulary from tokenized texts.

    Sort by frequency descending. Break ties alphabetically so results are deterministic.
    Return a set of tokens.
    """
    vocab = Counter()
    for tokenized_text in tokenized_texts:
        vocab.update(tokenized_text)
        
    return dict(vocab.most_common(max_size)).keys()


def encode_word_level(tokens, vocab, unk_token=UNK):
    """Replace tokens outside vocab with <unk>."""
    return [token if token in vocab else unk_token for token in tokens]

def oov_rate(tokens, vocab):
    """Fraction of tokens not present in vocab."""
    if not tokens:
        return 0.0
    return sum(t not in vocab for t in tokens) / len(tokens)

In [12]:
toy_vocab = build_word_vocab([["a", "b", "a", "c"]], max_size=2)
assert toy_vocab == {"a", "b"}, toy_vocab
assert encode_word_level(["a", "z"], toy_vocab) == ["a", UNK]
assert oov_rate(["a", "z"], toy_vocab) == 0.5
print("CHECK 3 passed - word vocabulary and OOV work")

CHECK 3 passed - word vocabulary and OOV work


In [13]:
train_regex = [regex_tokenize(t.lower()) for t in TRAIN_TEXTS]
test_regex = [regex_tokenize(t.lower()) for t in TEST_TEXTS]
test_tokens = [tok for text in test_regex for tok in text]

word_vocab = build_word_vocab(train_regex, max_size=25)
encoded = encode_word_level(test_tokens, word_vocab)

print("word vocab size:", len(word_vocab))
print("test tokens:    ", len(test_tokens))
print("OOV rate:       ", f"{100 * oov_rate(test_tokens, word_vocab):.1f}%")
print("encoded sample: ", encoded[:40])

word vocab size: 25
test tokens:     31
OOV rate:        64.5%
encoded sample:  ['the', '<unk>', 'sat', 'on', 'the', 'rug', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', 'new', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', 'new', 'york', '.', '<unk>', '<unk>', '<unk>', '12345', '<unk>', '<unk>', '<unk>', '.']


## BPE-Like Tokenization

Now implement the Day 2 `hug` example.

Training BPE means:

1. Start each word as characters.
2. Count adjacent pairs, weighted by word frequency.
3. Merge the most frequent pair.
4. Repeat.

This is a teaching implementation. When encoding a new word, it starts from that word's characters and replays the learned merges. Production byte-level tokenizers make the no-OOV guarantee stricter by starting from bytes instead of characters.

In [14]:
HUG_COUNTS = Counter({
    "hug": 10,
    "pug": 5,
    "pun": 12,
    "bun": 4,
    "hugs": 5,
})

def make_symbol_corpus(word_counts):
    """Map ('h','u','g') -> 10, etc."""
    return {tuple(word): count for word, count in word_counts.items()}

make_symbol_corpus(HUG_COUNTS)

{('h', 'u', 'g'): 10,
 ('p', 'u', 'g'): 5,
 ('p', 'u', 'n'): 12,
 ('b', 'u', 'n'): 4,
 ('h', 'u', 'g', 's'): 5}

## TODO 4 - Count Adjacent Pairs

For every tokenized word, count neighboring pairs. Remember to multiply by the word frequency.

In [15]:
def count_adjacent_pairs(symbol_corpus):
    """Count adjacent symbol pairs in a BPE symbol corpus.

    Args:
        symbol_corpus: dict mapping tuple(symbols) -> frequency

    Returns:
        Counter mapping (left_symbol, right_symbol) -> weighted count
    """
    pairs = Counter()
    for symbols, freq in symbol_corpus.items():
        for i in range(len(symbols)-1):
            pairs[(symbols[i],symbols[i+1])] += freq
    return pairs

In [16]:
pairs = count_adjacent_pairs(make_symbol_corpus(HUG_COUNTS))
assert pairs[("u", "g")] == 20, pairs
assert pairs[("p", "u")] == 17, pairs
assert pairs[("u", "n")] == 16, pairs
assert pairs.most_common(1)[0] == (("u", "g"), 20)
print("CHECK 4 passed - first BPE winner is ('u', 'g') with count 20")

CHECK 4 passed - first BPE winner is ('u', 'g') with count 20


## TODO 5 - Merge One Pair

Merging is a left-to-right scan. When you see the target pair, replace the two symbols with their concatenation and skip ahead by two.

In [17]:
def merge_pair_in_symbols(symbols, pair):
    """Merge one pair inside one tuple of symbols."""
    merged_symbols = []
    i = 0
    while i < len(symbols):
        cur = symbols[i]
        if i == len(symbols)-1:
            merged_symbols.append(cur)
            break
        
        next = symbols[i+1]

        if (cur, next) == pair:
            word = cur + next
            merged_symbols.append(word)
            i+=2
        else:
            merged_symbols.append(cur)
            i+=1
    return tuple(merged_symbols)

def merge_pair(symbol_corpus, pair):
    """Merge one pair everywhere in the symbol corpus."""
    after_merge = Counter()
    for symbols, count in symbol_corpus.items():
        new_symbols = merge_pair_in_symbols(symbols, pair)
        after_merge[new_symbols] = count
    return after_merge

In [18]:
assert merge_pair_in_symbols(("h", "u", "g", "s"), ("u", "g")) == ("h", "ug", "s")
after_one = merge_pair(make_symbol_corpus(HUG_COUNTS), ("u", "g"))
assert after_one[("h", "ug")] == 10
assert after_one[("p", "ug")] == 5
assert after_one[("h", "ug", "s")] == 5
print("CHECK 5 passed - merge works")

CHECK 5 passed - merge works


## TODO 6 - Train and Apply BPE

`train_bpe` is provided. It depends on your pair-counting and merge functions.

You implement `apply_bpe_to_word`: start from characters, replay learned merges in order.

In [ ]:
def train_bpe(word_counts, num_merges):
    symbol_corpus = make_symbol_corpus(word_counts)
    merges = []

    for _ in range(num_merges):
        pair_counts = count_adjacent_pairs(symbol_corpus)
        if not pair_counts:
            break
        best_pair, _ = pair_counts.most_common(1)[0]
        merges.append(best_pair)
        symbol_corpus = merge_pair(symbol_corpus, best_pair)

    vocab = set()
    for symbols in symbol_corpus:
        vocab.update(symbols)
    return merges, vocab, symbol_corpus

def apply_bpe_to_word(word, merges):
    """Tokenize one word by replaying BPE merges in order."""
    for pair in merges:
        word = merge_pair_in_symbols(word, pair)
    return list(word)

In [36]:
merges, bpe_vocab, final_corpus = train_bpe(HUG_COUNTS, num_merges=3)
assert merges == [("u", "g"), ("u", "n"), ("h", "ug")], merges
assert apply_bpe_to_word("bug", merges) == ["b", "ug"]
assert apply_bpe_to_word("hugs", merges) == ["hug", "s"]
print("CHECK 6 passed - BPE reproduces the Day 2 theory example")
print("merges:", merges)
print("bug ->", apply_bpe_to_word("bug", merges))
print("hugs ->", apply_bpe_to_word("hugs", merges))

CHECK 6 passed - BPE reproduces the Day 2 theory example
merges: [('u', 'g'), ('u', 'n'), ('h', 'ug')]
bug -> ['b', 'ug']
hugs -> ['hug', 's']


## Train Your BPE-Like Tokenizer on Text

This is intentionally simple. It treats regex word tokens as the units to train on, lowercases them, and applies BPE inside each token. Punctuation is kept as punctuation.

In [37]:
def has_alnum(token):
    return any(ch.isalnum() for ch in token)

def word_counts_from_texts(texts):
    counts = Counter()
    for text in texts:
        for tok in regex_tokenize(text.lower()):
            if has_alnum(tok):
                counts[tok] += 1
    return counts

def bpe_tokenize_text(text, merges):
    out = []
    for tok in regex_tokenize(text):
        if has_alnum(tok):
            out.extend(apply_bpe_to_word(tok.lower(), merges))
        else:
            out.append(tok)
    return out

toy_word_counts = word_counts_from_texts(TRAIN_TEXTS)
toy_merges, toy_bpe_vocab, _ = train_bpe(toy_word_counts, num_merges=40)

for text in TEST_TEXTS:
    print("TEXT:", text)
    print("BPE: ", bpe_tokenize_text(text, toy_merges))
    print()

TEXT: the bug sat on the rug
BPE:  ['the', 'b', 'ug', 'sat', 'on', 'the', 'rug']

TEXT: How many r's are in strawberry?
BPE:  ['h', 'o', 'w', 'm', 'an', 'y', 'r', "'", 's', 'a', 'r', 'e', 'i', 'n', 's', 't', 'r', 'a', 'w', 'be', 'r', 'r', 'y', '?']

TEXT: A new username vikrantGPT2026 appeared in New York.
BPE:  ['a', 'new', 'u', 's', 'e', 'r', 'n', 'a', 'm', 'e', 'v', 'i', 'k', 'r', 'an', 't', 'g', 'p', 't', '2', '0', '2', '6', 'a', 'p', 'p', 'e', 'a', 'r', 'e', 'd', 'i', 'n', 'new', 'york', '.']

TEXT: Try 987654 + 12345 without a calculator.
BPE:  ['t', 'r', 'y', '9', '8', '7', '6', '5', '4', '+', '1', '2', '3', '4', '5', 'w', 'i', 'th', 'o', 'u', 't', 'a', 'c', 'a', 'l', 'c', 'u', 'l', 'at', 'or', '.']



## Hugging Face `tokenizers`: Byte-Level BPE

Your BPE implementation is for learning. Production tokenizers use optimized libraries and more careful details.

This cell trains a tiny byte-level BPE tokenizer with Hugging Face `tokenizers`. Byte-level BPE starts from bytes, so it can represent any string, including emoji and non-English text.

In [38]:
try:
    from tokenizers import ByteLevelBPETokenizer

    hf_bpe = ByteLevelBPETokenizer()
    hf_bpe.train_from_iterator(
        TRAIN_TEXTS,
        vocab_size=300,
        min_frequency=1,
        special_tokens=["<pad>", "<unk>"],
    )

    hf_sample = "strawberry 12345 vikrantGPT2026 🙂 你好"
    enc = hf_bpe.encode(hf_sample)
    print("text:  ", hf_sample)
    print("tokens:", enc.tokens)
    print("ids:   ", enc.ids[:20], "...")
except Exception as exc:
    print("Hugging Face tokenizers demo skipped:", repr(exc))




text:   strawberry 12345 vikrantGPT2026 🙂 你好
tokens: ['s', 't', 'r', 'aw', 'b', 'er', 'r', 'y', 'Ġ1', '23', '45', 'Ġ', 'v', 'i', 'k', 'r', 'an', 't', 'G', 'P', 'T', '2', '0', '2', '6', 'Ġ', 'ð', 'Ł', 'Ļ', 'Ĥ', 'Ġ', 'ä', '½', 'ł', 'å', '¥', '½']
ids:    [84, 85, 83, 297, 67, 267, 83, 90, 278, 291, 292, 222, 87, 74, 76, 83, 266, 85, 40, 49] ...


## Optional Real Data: Tiny Shakespeare and WikiText-2

This cell tries to load:

- `tiny_shakespeare`
- `Salesforce/wikitext`, config `wikitext-2-raw-v1`

If download is unavailable, it uses local fallback text so the rest of the notebook still runs.

In [39]:
FALLBACK_SHAKESPEARE = """
First Citizen: Before we proceed any further, hear me speak.
All: Speak, speak.
First Citizen: You are all resolved rather to die than to famish?
All: Resolved. resolved.
"""

FALLBACK_WIKITEXT = """
Natural language processing is a field of artificial intelligence concerned with text.
Language models assign probabilities to sequences of tokens and can be evaluated on held-out data.
Tokenization changes the units used by the model and therefore changes token counts.
"""

def collect_text(dataset, max_rows=2000):
    pieces = []
    for i, row in enumerate(dataset):
        if i >= max_rows:
            break
        text = row.get("text", "")
        if text and text.strip():
            pieces.append(text)
    return "\n".join(pieces)

def load_practice_corpora():
    corpora = {
        "fallback_shakespeare": FALLBACK_SHAKESPEARE,
        "fallback_wikitext": FALLBACK_WIKITEXT,
    }

    try:
        from datasets import load_dataset

        tiny = load_dataset("tiny_shakespeare", split="train")
        tiny_text = collect_text(tiny)
        if tiny_text.strip():
            corpora["tiny_shakespeare"] = tiny_text
    except Exception as exc:
        print("Tiny Shakespeare download skipped:", repr(exc))

    try:
        from datasets import load_dataset

        wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        wiki_text = collect_text(wiki)
        if wiki_text.strip():
            corpora["wikitext2"] = wiki_text
    except Exception as exc:
        print("WikiText-2 download skipped:", repr(exc))

    return corpora

corpora = load_practice_corpora()
for name, text in corpora.items():
    print(f"{name:22} {len(text):8,} chars")

/Users/vikrant/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/vikrant/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tiny Shakespeare download skipped: RuntimeError('Dataset scripts are no longer supported, but found tiny_shakespeare.py')


Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 1111778.85 examples/s]

fallback_shakespeare        172 chars
fallback_wikitext           272 chars
wikitext2               627,469 chars


## Final Experiment - Compare Token Counts and OOV

For each corpus:

1. Train on the first chunk of text.
2. Test on the next chunk.
3. Compare whitespace, regex, word-level with `<unk>`, your BPE-like tokenizer, and Hugging Face byte-level BPE.

The exact numbers are less important than the pattern:

- Word-level tokenization has OOV.
- BPE-like tokenization splits rare words instead of collapsing them.
- Byte-level BPE can represent messy text robustly.
- Tokenizers produce different sequence lengths on the same text.

In [40]:
def hf_byte_bpe_tokens(train_text, test_text, vocab_size=1000):
    from tokenizers import ByteLevelBPETokenizer

    tok = ByteLevelBPETokenizer()
    tok.train_from_iterator(
        [train_text],
        vocab_size=max(vocab_size, 300),
        min_frequency=2,
        special_tokens=["<pad>", "<unk>"],
    )
    return tok.encode(test_text).tokens

def compare_tokenizers_on_text(text, train_chars=50_000, test_chars=10_000, word_vocab_size=500, bpe_merges=200):
    train_text = text[:train_chars]
    test_text = text[train_chars:train_chars + test_chars]
    if not test_text.strip():
        split = max(1, int(0.7 * len(text)))
        train_text = text[:split]
        test_text = text[split:]

    train_regex = [regex_tokenize(train_text.lower())]
    test_regex = regex_tokenize(test_text.lower())

    vocab = build_word_vocab(train_regex, max_size=word_vocab_size)
    word_encoded = encode_word_level(test_regex, vocab)

    word_counts = word_counts_from_texts([train_text])
    merges, _, _ = train_bpe(word_counts, num_merges=bpe_merges)
    bpe_tokens = bpe_tokenize_text(test_text, merges)

    rows = []
    rows.append(("whitespace", len(whitespace_tokenize(test_text)), None, whitespace_tokenize(test_text)[:20]))
    rows.append(("regex", len(test_regex), None, test_regex[:20]))
    rows.append(("word-level", len(word_encoded), oov_rate(test_regex, vocab), word_encoded[:20]))
    rows.append(("your BPE-like", len(bpe_tokens), None, bpe_tokens[:30]))

    try:
        hf_tokens = hf_byte_bpe_tokens(train_text, test_text)
        rows.append(("HF byte BPE", len(hf_tokens), 0.0, hf_tokens[:30]))
    except Exception as exc:
        rows.append(("HF byte BPE", "skipped", None, repr(exc)))

    return rows

for corpus_name, text in corpora.items():
    print("\n" + "=" * 80)
    print(corpus_name)
    print("=" * 80)

    rows = compare_tokenizers_on_text(text)
    for name, n_tokens, oov, sample_tokens in rows:
        oov_text = "" if oov is None else f" | OOV {100 * oov:.1f}%"
        print(f"{name:14} {str(n_tokens):>8} tokens{oov_text}")
        print("   sample:", sample_tokens)



fallback_shakespeare



whitespace            9 tokens
   sample: ['her', 'to', 'die', 'than', 'to', 'famish?', 'All:', 'Resolved.', 'resolved.']
regex                13 tokens
   sample: ['her', 'to', 'die', 'than', 'to', 'famish', '?', 'all', ':', 'resolved', '.', 'resolved', '.']
word-level           13 tokens | OOV 53.8%
   sample: ['<unk>', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', 'all', ':', 'resolved', '.', 'resolved', '.']
your BPE-like        26 tokens
   sample: ['h', 'e', 'r', 't', 'o', 'd', 'i', 'e', 't', 'h', 'an', 't', 'o', 'f', 'a', 'm', 'i', 's', 'h', '?', 'all', ':', 'resolved', '.', 'resolved', '.']
HF byte BPE          48 tokens | OOV 0.0%
   sample: ['h', 'e', 'r', 'Ġ', 't', 'o', 'Ġ', 'd', 'i', 'e', 'Ġ', 't', 'h', 'a', 'n', 'Ġ', 't', 'o', 'Ġ', 'f', 'a', 'm', 'i', 's', 'h', '?', 'Ċ', 'A', 'll', ':']

fallback_wikitext



whitespace           13 tokens
   sample: ['enization', 'changes', 'the', 'units', 'used', 'by', 'the', 'model', 'and', 'therefore', '

## Verified Results and Reflection

1. Where does whitespace tokenization fail most obviously?

Whitespace tokenization fails when punctuation or symbols are attached to words. In the Shakespeare fallback sample, it kept tokens like `famish?`, `All:`, and `resolved.` as single tokens. It also does not handle contractions, emoji, URLs, or languages without spaces well.

2. What did regex tokenization fix, and what did it still fail to solve?

Regex tokenization fixed punctuation splitting. For example, `famish?` became `famish` and `?`, and `resolved.` became `resolved` and `.`. But regex is still rule-based. It does not solve rare words, unknown words, or the deeper OOV problem.

3. Which test words became `<unk>` in the word-level tokenizer?

In the fallback Shakespeare sample, words like `her`, `to`, `die`, `than`, `famish`, and `?` became `<unk>`. In WikiText-2, many rare or unseen tokens became `<unk>`, including examples like `sussex`, `england`, `bequeathed`, `friend`, `edith`, and `andrew`.

4. How did your BPE-like tokenizer represent rare words instead?

The BPE-like tokenizer did not collapse rare words into `<unk>`. It broke them into smaller known pieces. For example, `sussex` became pieces like `su`, `s`, `se`, `x`, and `bequeathed` became smaller subword/character pieces. This is the key advantage of subword tokenization.

5. Which tokenizer produced the longest sequences?

It depended on the corpus. On the tiny fallback samples, Hugging Face byte-level BPE produced the longest sequences: 48 tokens for fallback Shakespeare and 65 for fallback WikiText. On the larger WikiText-2 sample, my BPE-like tokenizer was longest with 4,714 tokens, compared with 4,227 for Hugging Face byte-level BPE. The general lesson is that smaller or weaker subword vocabularies split more, so they create longer sequences.

6. Why does this make raw perplexity hard to compare across tokenizers?

Raw perplexity is measured per token, but each tokenizer defines tokens differently. A word tokenizer, regex tokenizer, BPE tokenizer, and byte-level tokenizer can produce different sequence lengths for the exact same text. So a perplexity number is only meaningful when the tokenizer is part of the comparison. For tokenizer-independent comparison, bits per character or bits per byte is safer.

7. What surprised you about `strawberry`, numbers, or non-English text?

The main surprise is that the model may not receive words as clean letters or numbers as clean digits. A word like `strawberry` can be represented as larger chunks, so counting letters like `r` is less natural for the model than it is for a human. Numbers and non-English text can also split into uneven pieces, which helps explain why these tasks can feel strange for LLMs.